# Phase 2 — stacked ensemble review

Reviews the per-region stacked ensemble (`bagpipe.models.stacked`, built on
the maintainer's [regional-stacker](https://github.com/GalKepler/regional-stacker)
package) on the same subject-grouped CV splits and bias-correction harness
as `baseline_model_demo.ipynb`, so the two are directly comparable. See
`docs/DESIGN.md` §4.1 for the harness design and `CLAUDE.md`'s Phase 2
status note for the model's current standing.

**Feature spec differs by design between the two model families** — this
was a real bug in the first cut of this notebook, caught by the
maintainer: the flat baselines (linear/ridge) get **GM volume only**
(`vol_gm`), one column per region — mixing GM/WM/CSF into one undifferentiated
flat vector would let e.g. a region's WM volume get treated as just another
feature indistinguishable from its GM volume, which isn't a meaningful flat
model. The stacked ensemble gets **all three metrics** (`vol_gm`, `vol_wm`,
`vol_csf`) grouped per region via `region_mapping` — its whole point is
fusing multiple metrics per region through a per-region base learner
*before* flattening to the meta-learner, which also makes per-region
importance inspectable (§4 below).

No subject IDs or raw feature values are printed below — only aggregate
counts and metrics. Outputs are stripped on commit (nbstripout) regardless.

In [ ]:
import numpy as np
import pandas as pd

from bagpipe.core.config import REPO_ROOT, get_path
from bagpipe.db.export_training_table import export as export_training_table
from bagpipe.models.tabular import build_region_matrix, build_region_mapping
from bagpipe.models.evaluate import evaluate
from bagpipe.models.bias_correction import get_corrector
from bagpipe.models.covariate_adjustment import TIVSexAdjustedRegressor
from bagpipe.models import baseline, stacked

from regional_stacker import RegionalStackingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV

## 1. Export + build the region-wise feature matrices

Same `globals.parquet`/`regional.parquet` export as the baseline notebook.
`build_region_matrix(..., metrics=...)` filters which per-region measures
go into the flat feature vector: `["vol_gm"]` for the flat baselines,
`["vol_gm", "vol_wm", "vol_csf"]` for the stacked ensemble.
`build_region_mapping` then groups the all-metrics columns into one block
per `atlas__region` — the input `RegionalStackingRegressor` needs for its
per-region base learners.

In [ ]:
export_summary = export_training_table()
pd.DataFrame(export_summary).T[["rows"]]

In [ ]:
X_gm, y_gm, groups_gm, region_columns_gm = build_region_matrix(
    get_path("datasets_dir"), metrics=["vol_gm"]
)
X_all, y_all, groups_all, region_columns_all = build_region_matrix(
    get_path("datasets_dir"), metrics=["vol_gm", "vol_wm", "vol_csf"]
)
region_mapping = build_region_mapping(region_columns_all)

print(
    f"GM-only: {X_gm.shape[0]} samples, {len(region_columns_gm)} region columns\n"
    f"All-metrics: {X_all.shape[0]} samples, {len(region_columns_all)} region columns, "
    f"{len(region_mapping)} distinct regions"
)

## 2. Run the eval harness — stacked ensemble vs. baselines

Identical grouped-CV split (`n_splits=5`) and Cole bias correction for all
three candidates — the split logic is the same, only the feature spec
differs per model family (see intro), so the leaderboard is still a fair
comparison of *modeling approach*, not of who got more data. The stacked
ensemble is deeper than the baselines (nested CV inside `evaluate`'s own
nested CV for bias correction), so this cell is the slow one — expect
several minutes.

In [ ]:
def stacker_fn():
    return RegionalStackingRegressor(
        region_mapping=region_mapping,
        base_estimator=RidgeCV(),
        meta_estimator=RidgeCV(),
        outer_cv=5,
        inner_cv=3,
        n_jobs=-1,
        random_state=0,
    )


candidates = {
    "linear": (lambda: LinearRegression(), X_gm, y_gm, groups_gm),
    "ridge": (lambda: RidgeCV(alphas=np.logspace(-3, 3, 13)), X_gm, y_gm, groups_gm),
    "stacked": (stacker_fn, X_all, y_all, groups_all),
}

results = {}
for name, (base_model_fn, Xc, yc, gc) in candidates.items():
    model_fn = lambda base_model_fn=base_model_fn: TIVSexAdjustedRegressor(base_model_fn)
    results[name] = evaluate(
        model_fn, Xc, yc, gc, n_splits=5, bias_corrector=get_corrector("cole")
    )

In [ ]:
leaderboard = pd.DataFrame({name: r.metrics for name, r in results.items()}).T
leaderboard.sort_values("mae_raw")

**Known tradeoff, not a bug**: `mae_corrected` is worse than `mae_raw` for
every model here, ridge included. Cole correction is fit to zero out the
BAG-age slope, not to minimize MAE — an affine correction that removes
age-dependent bias necessarily stretches predictions away from the
training-fold mean, which increases MAE on tails. Confirmed in §4 below via
the BAG-vs-age slope check. See `CLAUDE.md` Phase 2 status note.

## 3. Diagnostics — predictions, BAG, distributions, sex differences

Stacked model only, from here on. Same diagnostic battery as the baseline
notebook, for direct visual comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pred = results["stacked"].predictions.copy()
pred["sex"] = X_all[:, -1][pred["index"].to_numpy()]
pred["sex_label"] = pred["sex"].map({0.0: "Male", 1.0: "Female"})
pred["bag_raw"] = pred["y_pred_raw"] - pred["y_true"]
pred["bag_corrected"] = pred["y_pred_corrected"] - pred["y_true"]
print(f"{len(pred)} test predictions, stacked model")

### Predicted vs. true age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, col, title in zip(
    axes, ["y_pred_raw", "y_pred_corrected"], ["Raw predictions", "Cole-corrected predictions"]
):
    ax.scatter(pred["y_true"], pred[col], s=8, alpha=0.3)
    lims = [pred["y_true"].min(), pred["y_true"].max()]
    ax.plot(lims, lims, "k--", lw=1, label="y = x")
    ax.set_xlabel("True age")
    ax.set_ylabel("Predicted age")
    ax.set_title(f"{title} (stacked)")
    ax.legend()
plt.tight_layout()
plt.show()

### BAG vs. true age — the trend correction should remove

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["bag_raw", "bag_corrected"], ["Raw BAG", "Corrected BAG"]):
    ax.scatter(pred["y_true"], pred[col], s=8, alpha=0.3)
    slope, intercept, r, p, se = stats.linregress(pred["y_true"], pred[col])
    xs = np.array([pred["y_true"].min(), pred["y_true"].max()])
    ax.plot(xs, slope * xs + intercept, "r-", lw=2, label=f"slope={slope:.3f}, p={p:.1e}")
    ax.axhline(0, color="k", lw=1, ls=":")
    ax.set_xlabel("True age")
    ax.set_ylabel("BAG (predicted - true)")
    ax.set_title(f"{title} (stacked)")
    ax.legend()
plt.tight_layout()
plt.show()

### BAG distribution — raw vs. corrected

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.kdeplot(pred["bag_raw"], label="raw", ax=ax)
sns.kdeplot(pred["bag_corrected"], label="corrected", ax=ax)
ax.axvline(0, color="k", lw=1, ls=":")
ax.set_xlabel("BAG (years)")
ax.set_title("BAG distribution (stacked)")
ax.legend()
plt.show()

### Corrected BAG by sex

Cole correction is fit on true age only, not sex — a sex gap surviving it is
a genuine model finding, not an artifact of the correction. The ridge model
showed a real gap here (Female positive, Male negative, p<0.0001); worth
checking whether the stacked model reproduces it.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=pred, x="sex_label", y="bag_corrected", ax=ax)
sns.stripplot(data=pred, x="sex_label", y="bag_corrected", ax=ax, color="black", alpha=0.15, size=2)
ax.axhline(0, color="k", lw=1, ls=":")
ax.set_xlabel("Sex")
ax.set_ylabel("Corrected BAG (years)")
ax.set_title("Corrected BAG by sex (stacked)")
plt.show()

male_bag = pred.loc[pred["sex_label"] == "Male", "bag_corrected"]
female_bag = pred.loc[pred["sex_label"] == "Female", "bag_corrected"]
u_stat, p_value = stats.mannwhitneyu(male_bag, female_bag)
print(f"Male:   mean BAG={male_bag.mean():+.2f}y, median={male_bag.median():+.2f}y, n={len(male_bag)}")
print(f"Female: mean BAG={female_bag.mean():+.2f}y, median={female_bag.median():+.2f}y, n={len(female_bag)}")
print(f"Mann-Whitney U p-value: {p_value:.4f}")

## 4. Per-region diagnostics — what the stacked model gets that ridge doesn't

Fits one stacker on the **full** all-metrics dataset (not the CV harness)
purely to inspect its per-region structure via `regional_stacker`'s fitted
attributes: `region_cv_scores_` (each region's own outer-CV R² as a stage-1
base learner, fit on all three metrics jointly), `meta_best_params_`,
`meta_cv_scores_`. This is an in-sample / exploratory fit for
interpretability only — **not** a held-out metric, don't compare its
numbers to the leaderboard above.

In [ ]:
full_fit = TIVSexAdjustedRegressor(stacker_fn)
full_fit.fit(X_all, y_all)
stacker = full_fit.model_

region_scores = pd.Series(
    {name: scores.mean() for name, scores in stacker.region_cv_scores_.items()}
).sort_values(ascending=False)
print(f"meta-learner outer-CV R²: {stacker.meta_cv_scores_.mean():.3f} (+/- {stacker.meta_cv_scores_.std():.3f})")
print(f"meta-learner best params: {stacker.meta_best_params_}")
region_scores.describe()

### Top / bottom regions by base-learner CV R²

Which regions carry age signal (across GM+WM+CSF jointly) on their own vs.
which are near-noise for the meta-learner to (hopefully) downweight.

In [ ]:
top_n = 20
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
region_scores.head(top_n).iloc[::-1].plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title(f"Top {top_n} regions by base-learner CV R²")
axes[0].set_xlabel("R²")
region_scores.tail(top_n).iloc[::-1].plot.barh(ax=axes[1], color="indianred")
axes[1].set_title(f"Bottom {top_n} regions by base-learner CV R²")
axes[1].set_xlabel("R²")
plt.tight_layout()
plt.show()

## 5. MLflow logging

`stacked.run()` and `baseline.run()` are the config-driven wrappers the CLI
(`bag models train-stacked` / `bag models train-baseline`) calls — same
harness, plus MLflow logging to the shared local tracking store. Each
config now carries its own `features.metrics` (see `config/models/*.yaml`),
so the feature spec used is always logged alongside the metrics, not just
implied by which script ran.

In [ ]:
for module, config_name in [
    (baseline, "baseline_ridge.yaml"),
    (stacked, "stacked.yaml"),
]:
    result, info = module.run(REPO_ROOT / "config" / "models" / config_name)
    print(info["run_name"], {k: round(v, 3) for k, v in result.metrics.items()})

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{get_path('mlflow_dir') / 'mlflow.db'}")
runs = mlflow.search_runs(experiment_names=["bagpipe-baseline"])
runs[["tags.mlflow.runName", "params.metrics", "metrics.mae_raw", "metrics.mae_corrected", "metrics.r2_corrected"]]

## Next steps

See the leaderboard in §2 for current MAE numbers — re-run this notebook
for fresh ones rather than trusting stale figures copied into markdown.
Candidates worth trying next: LightGBM or non-linear base learners per
region (captures local nonlinearity ridge can't), a wider `base_param_grid`/
`meta_param_grid` via `stacked.yaml`, and checking whether the bottom-R²
regions above are worth dropping before stage 2. Per `docs/DESIGN.md` §7
Phase 2: SFCN fine-tune once the GPU driver is installed, then promote the
best model to `models_registry` as the v1 production model.